In [ ]:
COXETER_PARAMETERS = (3, 4, 5)   # Examples: (4, 4, 4) or (4, 4, "inf")

# Enter the complete L0.
LOW_0_WORDS = [
    "e", "1", "2", "3", "12", "13", "21", "23", "31", "32",
    "121", "131", "232", "313", "323", "2323", "3121", "3131",
    "3232", "23131", "23232", "123232",
]

# Enter the complete L1 (including all of L0 above); the program computes L1 minus L0 automatically.
LOW_1_WORDS = [
    "e", "1", "2", "3", "12", "13", "21", "23", "31", "32",
    "121", "131", "232", "313", "323", "2323", "3121", "3131",
    "3232", "23131", "23232", "123232",
    "123", "132", "213", "231", "312", "321", "1213", "1232",
    "1323", "2131", "2313", "12313", "12323", "13121", "13232",
    "23121", "123131", "213232", "323131", "1213232", "3123232",
]

IMAGE_SIZE = 2200                 # Use 2200 for posters or 700 for a quick preview
RENDER_DEPTH = 12                 # Usually unchanged; increase if the lines near the circumference are not dense enough
INCLUDE_COXETER_DIAGRAM = False   # False: omit the Coxeter triangle in the upper-left corner

DARK_PURPLE = (152, 149, 201)     # L0
LIGHT_PURPLE = (211, 209, 232)    # L1 minus L0
OUTPUT_FILE = None                # None selects an automatic filename; alternatively, use "my-arrangement.png"



"""Algebraic source of truth for rank-three hyperbolic Coxeter systems."""

from collections import deque
from dataclasses import dataclass, field
from fractions import Fraction
from itertools import combinations, product
from math import cos, inf, isinf, pi, sqrt
import re

import numpy as np


GENERATOR_ORDER = ("1", "2", "3")
INFINITE_ORDER = inf
INFINITE_ORDER_TOKENS = {"inf", "infinity", "∞", "unbounded"}


def normalize_coxeter_order(value):
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in INFINITE_ORDER_TOKENS:
            return INFINITE_ORDER
        try:
            value = int(normalized)
        except ValueError as exc:
            raise ValueError(f"Invalid Coxeter order: {value}") from exc
    if value == INFINITE_ORDER:
        return INFINITE_ORDER
    if not isinstance(value, int) or isinstance(value, bool) or value < 2:
        raise ValueError("A Coxeter order must be an integer at least 2 or infinity.")
    return value


def coxeter_order_reciprocal(value):
    value = normalize_coxeter_order(value)
    return Fraction(0, 1) if isinf(value) else Fraction(1, value)


def coxeter_order_cosine(value):
    value = normalize_coxeter_order(value)
    return 1.0 if isinf(value) else cos(pi / value)


def coxeter_order_label(value):
    value = normalize_coxeter_order(value)
    return "∞" if isinf(value) else str(value)


def serialized_coxeter_order(value):
    value = normalize_coxeter_order(value)
    return "inf" if isinf(value) else value


def lorentz_normals_from_gram(gram, tolerance=1e-9):
    """Factor a signature-(2,1) Gram matrix into Lorentzian normals."""
    gram = np.asarray(gram, dtype=np.float64)
    eigenvalues, eigenvectors = np.linalg.eigh(gram)
    negative = [index for index, value in enumerate(eigenvalues) if value < -tolerance]
    positive = [index for index, value in enumerate(eigenvalues) if value > tolerance]
    if len(negative) != 1 or len(positive) != 2:
        raise ValueError("The Gram matrix does not have Lorentz signature (2,1).")
    order = negative + positive
    ordered_values = eigenvalues[order]
    ordered_vectors = eigenvectors[:, order]
    normal_matrix = np.diag(np.sqrt(np.abs(ordered_values))) @ ordered_vectors.T

    minkowski = np.diag((-1.0, 1.0, 1.0))
    equations = normal_matrix.T @ minkowski
    center = np.linalg.solve(equations, -np.ones(3))
    norm = float(center @ minkowski @ center)
    if norm >= -tolerance:
        raise ArithmeticError("The center of the fundamental chamber is not on the hyperboloid.")
    if center[0] < 0:
        normal_matrix[0, :] *= -1.0

    np.testing.assert_allclose(
        normal_matrix.T @ minkowski @ normal_matrix,
        gram,
        atol=max(tolerance * 10, 1e-10),
    )
    return tuple(normal_matrix[:, index].copy() for index in range(3))


@dataclass
class RootRecord:
    key: tuple[int, ...]
    coefficients: np.ndarray
    reflection: np.ndarray
    depth: int
    is_small: bool = False


@dataclass
class WallRecord:
    root_key: tuple[int, ...]
    reflection: np.ndarray
    wall_type: str
    chamber_key: tuple[int, ...]
    neighbor_key: tuple[int, ...]


@dataclass
class ChamberRecord:
    key: tuple[int, ...]
    word: str
    matrix: np.ndarray
    aliases: set[str] = field(default_factory=set)
    length: int = 0
    inversions: tuple[tuple[int, ...], ...] = ()
    walls: dict[str, WallRecord] = field(default_factory=dict)


class CoxeterSystem3:
    """The geometric representation of a rank-three triangle Coxeter group.

    Roots use column coordinates in the ordered simple-root basis
    ``(alpha_1, alpha_2, alpha_3)``.  A displayed word ``123`` has matrix
    ``R_1 @ R_2 @ R_3``.
    """

    def __init__(self, m12, m13, m23, tolerance=1e-9):
        parameters = tuple(
            normalize_coxeter_order(value)
            for value in (m12, m13, m23)
        )
        if sum(coxeter_order_reciprocal(value) for value in parameters) >= 1:
            raise ValueError("The parameters do not satisfy the hyperbolicity condition: 1/m12 + 1/m13 + 1/m23 < 1.")

        self.m12, self.m13, self.m23 = parameters
        self.tolerance = float(tolerance)
        self.max_element_residual = 0.0
        self.max_root_residual = 0.0
        self._element_buckets = {}
        self._element_entries = []
        self._root_key_buckets = {}
        self._root_key_entries = []
        self.gram = np.array(
            [
                [
                    1.0,
                    -coxeter_order_cosine(self.m12),
                    -coxeter_order_cosine(self.m13),
                ],
                [
                    -coxeter_order_cosine(self.m12),
                    1.0,
                    -coxeter_order_cosine(self.m23),
                ],
                [
                    -coxeter_order_cosine(self.m13),
                    -coxeter_order_cosine(self.m23),
                    1.0,
                ],
            ],
            dtype=np.float64,
        )

        eigenvalues = np.linalg.eigvalsh(self.gram)
        positive = int(np.count_nonzero(eigenvalues > self.tolerance))
        negative = int(np.count_nonzero(eigenvalues < -self.tolerance))
        zero = 3 - positive - negative
        self.signature = (positive, negative, zero)
        if self.signature != (2, 1, 0):
            raise ValueError(
                "The Gram matrix does not have rank-three hyperbolic signature (2,1): "
                f"received {self.signature}."
            )

        identity = np.eye(3, dtype=np.float64)
        self.simple_roots = tuple(identity[:, index].copy() for index in range(3))
        self.generator_matrices = {}
        for index, generator in enumerate(GENERATOR_ORDER):
            simple_root = self.simple_roots[index]
            self.generator_matrices[generator] = identity - 2.0 * np.outer(
                simple_root,
                self.gram[index, :],
            )
        self._root_registry = {}

    @staticmethod
    def _normalize_word(word):
        normalized = str(word).strip().lower()
        if normalized == "e":
            return ""
        if not re.fullmatch(r"[123]*", normalized):
            raise ValueError(f"Invalid Coxeter word: {word}")
        return normalized

    def word_matrix(self, word):
        matrix = np.eye(3, dtype=np.float64)
        for generator in self._normalize_word(word):
            matrix = matrix @ self.generator_matrices[generator]
        return matrix

    def element_key(self, word_or_matrix):
        matrix = (
            self.word_matrix(word_or_matrix)
            if isinstance(word_or_matrix, str)
            else np.asarray(word_or_matrix, dtype=np.float64)
        )
        if matrix.shape != (3, 3):
            raise ValueError("A group element matrix must be 3x3.")
        coordinates = self._key_coordinates(matrix)
        quantum = max(self.tolerance * 1000, 1e-6)
        bucket = tuple(
            int(value)
            for value in np.rint(coordinates / quantum).astype(np.int64)
        )
        candidates = self._bucket_candidates(
            coordinates,
            quantum,
            self._element_buckets,
        )
        match = self._find_residual_match(matrix, candidates, is_root=False)
        if match is not None:
            primary = self._element_buckets.setdefault(bucket, [])
            if not any(existing is match for existing in primary):
                primary.append(match)
            return match[1]

        key = bucket + (len(self._element_entries),)
        entry = (matrix.copy(), key)
        self._element_buckets.setdefault(bucket, []).append(entry)
        self._element_entries.append(entry)
        return key

    @staticmethod
    def _key_coordinates(value):
        flattened = np.asarray(value, dtype=np.float64).reshape(-1)
        scale = max(1.0, float(np.max(np.abs(flattened))))
        return np.concatenate((flattened / scale, (np.log(scale),)))

    def _bucket_candidates(self, value, quantum, buckets):
        options = []
        threshold = max(self.tolerance * 20, 1e-8)
        for coordinate in np.asarray(value).reshape(-1):
            scaled = float(coordinate) / quantum
            primary = int(np.rint(scaled))
            coordinate_options = [primary]
            if (scaled - (primary - 0.5)) * quantum <= threshold:
                coordinate_options.append(primary - 1)
            if ((primary + 0.5) - scaled) * quantum <= threshold:
                coordinate_options.append(primary + 1)
            options.append(tuple(dict.fromkeys(coordinate_options)))

        candidates = []
        seen = set()
        for bucket in product(*options):
            for entry in buckets.get(tuple(bucket), ()):
                identity = id(entry)
                if identity not in seen:
                    seen.add(identity)
                    candidates.append(entry)
        return candidates

    def _find_residual_match(self, value, entries, is_root):
        if not entries:
            return None
        values = np.stack([entry[0] for entry in entries])
        absolute_residuals = np.max(
            np.abs(values - value),
            axis=tuple(range(1, values.ndim)),
        )
        entry_scales = np.max(
            np.abs(values),
            axis=tuple(range(1, values.ndim)),
        )
        value_scale = float(np.max(np.abs(value)))
        normalized_residuals = absolute_residuals / (
            1.0 + np.maximum(entry_scales, value_scale)
        )
        close = normalized_residuals <= self.tolerance * 10
        indexes = np.flatnonzero(close)
        if not indexes.size:
            return None
        match_index = int(indexes[np.argmin(normalized_residuals[indexes])])
        entry = entries[match_index]
        residual = float(normalized_residuals[match_index])
        if is_root:
            self.max_root_residual = max(self.max_root_residual, residual)
        else:
            self.max_element_residual = max(self.max_element_residual, residual)
        return entry

    def same_element(self, left, right):
        left_matrix = self.word_matrix(left)
        right_matrix = self.word_matrix(right)
        residual = np.max(np.abs(left_matrix - right_matrix)) / (
            1.0
            + max(
                np.max(np.abs(left_matrix)),
                np.max(np.abs(right_matrix)),
            )
        )
        return bool(residual <= self.tolerance * 10)

    def _positive_root(self, root):
        coefficients = np.asarray(root, dtype=np.float64).reshape(3).copy()
        coefficients[np.abs(coefficients) <= self.tolerance * 10] = 0.0
        if np.all(coefficients >= 0.0):
            return coefficients
        if np.all(coefficients <= 0.0):
            return -coefficients
        raise ArithmeticError(
            "The computed real root is neither positive nor negative; the generator order or numerical error is inconsistent."
        )

    def root_key(self, root):
        positive = self._positive_root(root)
        coordinates = self._key_coordinates(positive)
        quantum = max(self.tolerance * 1000, 1e-6)
        bucket = tuple(
            int(value)
            for value in np.rint(coordinates / quantum).astype(np.int64)
        )
        candidates = self._bucket_candidates(
            coordinates,
            quantum,
            self._root_key_buckets,
        )
        match = self._find_residual_match(positive, candidates, is_root=True)
        if match is not None:
            primary = self._root_key_buckets.setdefault(bucket, [])
            if not any(existing is match for existing in primary):
                primary.append(match)
            return match[1]

        key = bucket + (len(self._root_key_entries),)
        entry = (positive.copy(), key)
        self._root_key_buckets.setdefault(bucket, []).append(entry)
        self._root_key_entries.append(entry)
        return key

    def reflection_for_root(self, root):
        beta = np.asarray(root, dtype=np.float64).reshape(3)
        norm = float(beta.T @ self.gram @ beta)
        if abs(norm) <= self.tolerance:
            raise ValueError("An isotropic vector cannot define a Coxeter reflection.")
        return np.eye(3) - (2.0 / norm) * np.outer(beta, beta.T @ self.gram)

    def _record_root(self, root, depth):
        positive = self._positive_root(root)
        key = self.root_key(positive)
        record = self._root_registry.get(key)
        if record is None:
            record = RootRecord(
                key=key,
                coefficients=positive,
                reflection=self.reflection_for_root(positive),
                depth=int(depth),
            )
            self._root_registry[key] = record
        else:
            record.depth = min(record.depth, int(depth))
        return record

    def enumerate_chambers(self, max_length):
        if not isinstance(max_length, int) or max_length < 0:
            raise ValueError("max_length must be a nonnegative integer.")

        identity = np.eye(3)
        identity_key = self.element_key(identity)
        chambers = {
            identity_key: ChamberRecord(
                key=identity_key,
                word="e",
                matrix=identity.copy(),
                aliases={"e"},
                length=0,
            )
        }
        queue = deque([identity_key])

        while queue:
            chamber_key = queue.popleft()
            chamber = chambers[chamber_key]
            word_digits = "" if chamber.word == "e" else chamber.word
            matrix = chamber.matrix
            matrix_inverse = np.linalg.inv(matrix)

            for index, generator in enumerate(GENERATOR_ORDER):
                candidate_word = word_digits + generator
                candidate_matrix = matrix @ self.generator_matrices[generator]
                candidate_key = self.element_key(candidate_matrix)
                wall_root = matrix @ self.simple_roots[index]
                root_record = self._record_root(
                    wall_root,
                    depth=chamber.length + 1,
                )
                wall_reflection = (
                    matrix
                    @ self.generator_matrices[generator]
                    @ matrix_inverse
                )
                chamber.walls[generator] = WallRecord(
                    root_key=root_record.key,
                    reflection=wall_reflection,
                    wall_type=generator,
                    chamber_key=chamber.key,
                    neighbor_key=candidate_key,
                )

                existing = chambers.get(candidate_key)
                if existing is not None:
                    existing.aliases.add(candidate_word or "e")
                    continue
                if chamber.length >= max_length:
                    continue

                candidate = ChamberRecord(
                    key=candidate_key,
                    word=candidate_word,
                    matrix=candidate_matrix.copy(),
                    aliases={candidate_word},
                    length=chamber.length + 1,
                    inversions=chamber.inversions + (root_record.key,),
                )
                chambers[candidate_key] = candidate
                queue.append(candidate_key)

        return chambers

    def canonicalize_word(self, word):
        normalized = self._normalize_word(word)
        display_word = normalized or "e"
        chambers = self.enumerate_chambers(len(normalized))
        key = self.element_key(self.word_matrix(normalized))
        chamber = chambers.get(key)
        if chamber is None:
            target = self.word_matrix(normalized)
            chamber = next(
                (
                    record
                    for record in chambers.values()
                    if np.allclose(
                        self.word_matrix(record.word),
                        target,
                        atol=self.tolerance * 10,
                        rtol=self.tolerance * 10,
                    )
                ),
                None,
            )
        if chamber is None:
            raise ArithmeticError(f"Cannot find the chamber corresponding to the word {word}.")
        chamber.aliases.add(display_word)
        return chamber

    def enumerate_small_roots(self):
        small = {}
        queue = deque()

        for simple_root in self.simple_roots:
            record = self._record_root(simple_root, depth=1)
            if record.key not in small:
                record.is_small = True
                small[record.key] = record
                queue.append(record)

        while queue:
            record = queue.popleft()
            beta = record.coefficients
            for index, generator in enumerate(GENERATOR_ORDER):
                inner_product = float(
                    beta.T @ self.gram @ self.simple_roots[index]
                )
                if not (-1.0 + self.tolerance < inner_product < -self.tolerance):
                    continue
                reflected = self.generator_matrices[generator] @ beta
                candidate = self._record_root(reflected, depth=record.depth + 1)
                if candidate.key in small:
                    continue
                candidate.is_small = True
                small[candidate.key] = candidate
                queue.append(candidate)

        return small

    def enumerate_m_small_roots(self, m):
        """Enumerate Sigma_m using the dominance-depth reflection rule."""
        if not isinstance(m, int) or m < 0:
            raise ValueError("m must be a nonnegative integer.")
        if m == 0:
            return self.enumerate_small_roots()

        roots = dict(self.enumerate_small_roots())
        dominance_depths = {key: 0 for key in roots}
        queue = deque(roots)

        while queue:
            key = queue.popleft()
            record = roots[key]
            beta = record.coefficients
            current_depth = dominance_depths[key]
            for index, generator in enumerate(GENERATOR_ORDER):
                reflected = self.generator_matrices[generator] @ beta
                if np.all(reflected <= self.tolerance * 10):
                    continue
                reflected = self._positive_root(reflected)
                candidate_key = self.root_key(reflected)
                inner_product = float(
                    beta.T @ self.gram @ self.simple_roots[index]
                )
                comparison_tolerance = self.tolerance * 100
                if inner_product >= 1.0 - comparison_tolerance:
                    candidate_depth = current_depth - 1
                elif inner_product <= -1.0 + comparison_tolerance:
                    candidate_depth = current_depth + 1
                else:
                    candidate_depth = current_depth

                if candidate_depth < 0 or candidate_depth > m:
                    continue
                known_depth = dominance_depths.get(candidate_key)
                if known_depth is not None:
                    if known_depth != candidate_depth:
                        raise ArithmeticError(
                            "Inconsistent dominance depths were obtained for the same root."
                        )
                    continue

                candidate = self._record_root(
                    reflected,
                    depth=record.depth + 1,
                )
                roots[candidate_key] = candidate
                dominance_depths[candidate_key] = candidate_depth
                queue.append(candidate_key)

        return roots

    def cone_contains(self, root, generators):
        target = np.asarray(root, dtype=np.float64).reshape(3)
        candidates = [
            np.asarray(generator, dtype=np.float64).reshape(3)
            for generator in generators
        ]
        if np.linalg.norm(target) <= self.tolerance:
            return True
        if not candidates:
            return False

        residual_tolerance = max(self.tolerance * 100, 1e-8) * (
            1.0 + np.linalg.norm(target)
        )
        for size in range(1, min(3, len(candidates)) + 1):
            for selected in combinations(candidates, size):
                matrix = np.column_stack(selected)
                coefficients, _, _, _ = np.linalg.lstsq(matrix, target, rcond=None)
                residual = np.linalg.norm(matrix @ coefficients - target)
                if (
                    residual <= residual_tolerance
                    and np.all(coefficients >= -self.tolerance * 100)
                ):
                    return True
        return False

    def is_low_element(self, chamber, small_roots=None):
        small = small_roots or self.enumerate_small_roots()
        if not chamber.inversions:
            return True

        inversion_records = [
            self._root_registry[root_key]
            for root_key in chamber.inversions
        ]
        small_inversions = [
            record.coefficients
            for record in inversion_records
            if record.key in small
        ]
        return all(
            self.cone_contains(record.coefficients, small_inversions)
            for record in inversion_records
        )

    def enumerate_low_elements(self):
        """Enumerate L_0 using its suffix-closed Garside-shadow property.

        Every non-identity low element has a reduced expression whose suffix
        after deleting the first generator is again low.  Starting at the
        identity and prepending generators therefore terminates exactly when
        the finite set L_0 is exhausted.
        """

        small_roots = self.enumerate_small_roots()
        identity = self.canonicalize_word("e")
        low_elements = {identity.key: identity}
        frontier = [identity]

        while frontier:
            next_frontier = []
            for chamber in frontier:
                suffix = "" if chamber.word == "e" else chamber.word
                for generator in GENERATOR_ORDER:
                    candidate = self.canonicalize_word(generator + suffix)
                    if candidate.length != chamber.length + 1:
                        continue
                    if candidate.key in low_elements:
                        continue
                    if not self.is_low_element(candidate, small_roots):
                        continue
                    low_elements[candidate.key] = candidate
                    next_frontier.append(candidate)
            frontier = next_frontier

        return low_elements

    def enumerate_m_low_elements(self, m):
        """Enumerate the finite Garside shadow L_m."""
        if not isinstance(m, int) or m < 0:
            raise ValueError("m must be a nonnegative integer.")
        if m == 0:
            return self.enumerate_low_elements()

        m_small_roots = self.enumerate_m_small_roots(m)
        identity = self.canonicalize_word("e")
        low_elements = {identity.key: identity}
        frontier = [identity]

        while frontier:
            next_frontier = []
            for chamber in frontier:
                suffix = "" if chamber.word == "e" else chamber.word
                for generator in GENERATOR_ORDER:
                    candidate = self.canonicalize_word(generator + suffix)
                    if candidate.length != chamber.length + 1:
                        continue
                    if candidate.key in low_elements:
                        continue
                    if not self.is_low_element(candidate, m_small_roots):
                        continue
                    low_elements[candidate.key] = candidate
                    next_frontier.append(candidate)
            frontier = next_frontier

        return low_elements


"""Analytic, paper-style rendering for rank-three Coxeter arrangements."""

from dataclasses import dataclass
from functools import lru_cache
from math import atan2, cos, isinf, pi, sin, sqrt
from pathlib import Path
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont



MINKOWSKI_FORM = np.diag((-1.0, 1.0, 1.0))


@dataclass(frozen=True)
class RenderStyle:
    fill_color: tuple[int, int, int] = (152, 149, 201)
    secondary_fill_color: tuple[int, int, int] = (211, 209, 232)
    identity_fill_color: tuple[int, int, int] = (166, 166, 166)
    small_wall_color: tuple[int, int, int] = (67, 69, 164)
    emphasized_wall_color: tuple[int, int, int] = (0, 0, 0)
    grid_color: tuple[int, int, int] = (28, 28, 28)
    context_label_color: tuple[int, int, int] = (145, 145, 145)
    supersample: int = 3
    grid_width: float = 1.0
    small_wall_width: float = 2.2
    emphasized_wall_width: float = 3.2


@dataclass(frozen=True)
class Geodesic:
    kind: str
    center: tuple[float, float] | None
    radius: float | None
    endpoints: tuple[tuple[float, float], tuple[float, float]]

    def is_orthogonal_to_boundary(self, atol=1e-8):
        if self.kind == "diameter":
            return True
        center_squared = self.center[0] ** 2 + self.center[1] ** 2
        return abs(center_squared - self.radius**2 - 1.0) <= atol

    def sample(self, samples=181):
        start, stop = self.endpoints
        if self.kind == "diameter":
            parameter = np.linspace(0.0, 1.0, samples)
            return np.column_stack(
                (
                    start[0] + parameter * (stop[0] - start[0]),
                    start[1] + parameter * (stop[1] - start[1]),
                )
            )

        cx, cy = self.center
        first_angle = atan2(start[1] - cy, start[0] - cx)
        second_angle = atan2(stop[1] - cy, stop[0] - cx)
        delta = (second_angle - first_angle + pi) % (2 * pi) - pi
        angles = np.linspace(first_angle, first_angle + delta, samples)
        return np.column_stack(
            (
                cx + self.radius * np.cos(angles),
                cy + self.radius * np.sin(angles),
            )
        )


@dataclass(frozen=True)
class RenderResult:
    image_path: Path
    label_positions: dict[str, tuple[int, int]]
    label_font_sizes: dict[str, float]
    label_rotations: dict[str, int]
    root_keys: tuple[tuple[int, ...], ...]
    small_root_keys: tuple[tuple[int, ...], ...]
    emphasized_root_keys: tuple[tuple[int, ...], ...]
    root_records: tuple["RenderedRootRecord", ...]
    wall_occurrences: tuple["RenderedWallOccurrence", ...]
    rendered_chamber_keys: tuple[tuple[int, ...], ...]
    curve_draw_counts: dict[tuple[int, ...], int]
    small_curve_draw_counts: dict[tuple[int, ...], int]
    emphasized_curve_draw_counts: dict[tuple[int, ...], int]
    visible_curve_keys: tuple[tuple[int, ...], ...]
    visible_small_curve_keys: tuple[tuple[int, ...], ...]
    visible_emphasized_curve_keys: tuple[tuple[int, ...], ...]
    low_chamber_keys: tuple[tuple[int, ...], ...]
    secondary_chamber_keys: tuple[tuple[int, ...], ...]
    effective_render_depth: int
    canvas_metrics: dict
    elapsed_seconds: float


@dataclass(frozen=True)
class RenderedRootRecord:
    key: tuple[int, ...]
    coefficients: np.ndarray
    reflection: np.ndarray
    occurrences: int
    is_small: bool


@dataclass(frozen=True)
class RenderedWallOccurrence:
    chamber_key: tuple[int, ...]
    chamber_word: str
    wall_type: str
    root_key: tuple[int, ...]
    reflection: np.ndarray
    neighbor_key: tuple[int, ...]
    curve_key: tuple[int, ...]


def _lorentz_normals(system):
    if isinf(system.m23):
        return lorentz_normals_from_gram(system.gram, system.tolerance)

    angle_23 = pi / system.m23
    normal_2 = np.array((0.0, 1.0, 0.0))
    normal_3 = np.array((0.0, -cos(angle_23), sin(angle_23)))

    x_1 = -cos(pi / system.m12)
    y_1 = (
        -cos(pi / system.m13) - normal_3[1] * x_1
    ) / normal_3[2]
    t_1 = sqrt(max(0.0, x_1 * x_1 + y_1 * y_1 - 1.0))
    normal_1 = np.array((t_1, x_1, y_1))
    normals = (normal_1, normal_2, normal_3)

    computed_gram = np.array(
        [
            [left @ MINKOWSKI_FORM @ right for right in normals]
            for left in normals
        ]
    )
    np.testing.assert_allclose(computed_gram, system.gram, atol=1e-8)
    return normals


def _normal_for_root(root, system):
    coefficients = (
        root.coefficients
        if isinstance(root, RootRecord)
        else np.asarray(root, dtype=np.float64).reshape(3)
    )
    normal_matrix = np.column_stack(_lorentz_normals(system))
    return normal_matrix @ coefficients


def poincare_geodesic(root, system):
    b0, b1, b2 = _normal_for_root(root, system)
    tolerance = system.tolerance * 100
    if abs(b0) <= tolerance:
        length = sqrt(b1 * b1 + b2 * b2)
        if length <= tolerance:
            raise ValueError("A zero normal vector cannot define a Poincare geodesic.")
        direction = np.array((-b2 / length, b1 / length))
        return Geodesic(
            kind="diameter",
            center=None,
            radius=None,
            endpoints=(tuple(-direction), tuple(direction)),
        )

    center = np.array((b1 / b0, b2 / b0), dtype=np.float64)
    center_length = float(np.linalg.norm(center))
    radius = sqrt(max(0.0, center_length * center_length - 1.0))
    foot = center / (center_length * center_length)
    height = sqrt(max(0.0, 1.0 - 1.0 / (center_length * center_length)))
    perpendicular = np.array((-center[1], center[0])) / center_length
    first = foot + height * perpendicular
    second = foot - height * perpendicular
    return Geodesic(
        kind="circle",
        center=tuple(center),
        radius=radius,
        endpoints=(tuple(first), tuple(second)),
    )


def _lorentz_reflection(normal):
    return np.eye(3) - 2.0 * np.outer(normal, normal @ MINKOWSKI_FORM)


def _fundamental_center(system, normals):
    equations = np.array([normal @ MINKOWSKI_FORM for normal in normals])
    center = np.linalg.solve(equations, -np.ones(3))
    norm = float(center @ MINKOWSKI_FORM @ center)
    if norm >= 0:
        raise ArithmeticError("The center of the fundamental chamber is not on the hyperboloid.")
    center /= sqrt(-norm)
    if center[0] < 0:
        center = -center
    return center


def _word_center(system, word, normals, fundamental_center):
    matrices = {
        generator: _lorentz_reflection(normals[index])
        for index, generator in enumerate(GENERATOR_ORDER)
    }
    matrix = np.eye(3)
    normalized = "" if str(word).lower() == "e" else str(word)
    for generator in normalized:
        matrix = matrix @ matrices[generator]
    point = matrix @ fundamental_center
    return np.array((point[1] / (point[0] + 1), point[2] / (point[0] + 1)))


def _orient(points, rotation, reflect):
    oriented = np.asarray(points, dtype=np.float64).copy()
    if reflect:
        oriented[..., 0] *= -1
    rotation_matrix = np.array(
        ((cos(rotation), -sin(rotation)), (sin(rotation), cos(rotation)))
    )
    return oriented @ rotation_matrix.T


@lru_cache(maxsize=None)
def _load_font(size):
    for name in ("times.ttf", "DejaVuSerif.ttf", "arial.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default()


def _draw_rotated_text(image, position, text, font, fill, rotation):
    if rotation == 0:
        ImageDraw.Draw(image).text(
            position,
            text,
            fill=fill,
            font=font,
            anchor="mm",
        )
        return

    left, top, right, bottom = font.getbbox(text)
    padding = 2
    text_image = Image.new(
        "RGBA",
        (right - left + 2 * padding, bottom - top + 2 * padding),
        (255, 255, 255, 0),
    )
    ImageDraw.Draw(text_image).text(
        (padding - left, padding - top),
        text,
        fill=fill,
        font=font,
    )
    text_image = text_image.rotate(rotation, expand=True)
    paste_position = (
        round(position[0] - text_image.width / 2),
        round(position[1] - text_image.height / 2),
    )
    image.paste(text_image, paste_position, text_image)


def _nearest_white_pixel(image, seed, radius=10):
    width, height = image.size
    x, y = seed
    for distance in range(radius + 1):
        for dy in range(-distance, distance + 1):
            dx = distance - abs(dy)
            for candidate_x in ({x + dx, x - dx} if dx else {x}):
                candidate_y = y + dy
                if (
                    0 <= candidate_x < width
                    and 0 <= candidate_y < height
                    and image.getpixel((candidate_x, candidate_y)) == (255, 255, 255)
                ):
                    return candidate_x, candidate_y
    raise ValueError("The chamber center lies on a line; cannot find a safe seed for filling.")


def _diagram_vertices(canvas_width, canvas_height):
    return {
        "1": (0.135 * canvas_width, 0.115 * canvas_height),
        "3": (0.095 * canvas_width, 0.215 * canvas_height),
        "2": (0.175 * canvas_width, 0.215 * canvas_height),
    }


def _distance_to_segment(point, start, stop):
    point = np.asarray(point, dtype=np.float64)
    start = np.asarray(start, dtype=np.float64)
    stop = np.asarray(stop, dtype=np.float64)
    direction = stop - start
    length_squared = float(direction @ direction)
    if length_squared == 0:
        return float(np.linalg.norm(point - start))
    parameter = float(np.clip((point - start) @ direction / length_squared, 0, 1))
    return float(np.linalg.norm(point - (start + parameter * direction)))


def _diagram_is_outside_disk(canvas_width, canvas_height, disk_center, disk_radius, scale):
    vertices = _diagram_vertices(canvas_width, canvas_height)
    edges = (("1", "2"), ("1", "3"), ("2", "3"))
    clearance = max(4, round(5 * scale))
    minimum_distance = min(
        _distance_to_segment(disk_center, vertices[left], vertices[right])
        for left, right in edges
    )
    return minimum_distance - clearance > disk_radius


def _diagram_edge_label_position(edge, vertices, canvas_width, canvas_height):
    left, right = edge
    start = vertices[left]
    stop = vertices[right]
    midpoint = ((start[0] + stop[0]) / 2, (start[1] + stop[1]) / 2)
    offsets = {
        ("1", "3"): (-0.012 * canvas_width, -0.010 * canvas_height),
        ("1", "2"): (0.012 * canvas_width, -0.010 * canvas_height),
        ("2", "3"): (0.0, 0.015 * canvas_height),
    }
    dx, dy = offsets[edge]
    return midpoint[0] + dx, midpoint[1] + dy


def _draw_coxeter_diagram(draw, system, canvas_width, canvas_height, scale):
    vertices = _diagram_vertices(canvas_width, canvas_height)
    edge_orders = {("1", "2"): system.m12, ("1", "3"): system.m13, ("2", "3"): system.m23}
    line_width = max(6, round(5.2 * scale))
    font = _load_font(max(18, round(32 * scale)))
    for (left, right), order in edge_orders.items():
        start = vertices[left]
        stop = vertices[right]
        draw.line((start, stop), fill=(30, 30, 30), width=line_width)
        if order != 3:
            label_position = _diagram_edge_label_position(
                (left, right),
                vertices,
                canvas_width,
                canvas_height,
            )
            draw.text(
                label_position,
                coxeter_order_label(order),
                fill=(35, 35, 35),
                font=font,
                anchor="mm",
            )
    node_radius = max(4, round(6.0 * scale))
    label_offsets = {"1": (0, -0.020), "3": (-0.014, 0.020), "2": (0.014, 0.020)}
    for generator, (x, y) in vertices.items():
        draw.ellipse(
            (x - node_radius, y - node_radius, x + node_radius, y + node_radius),
            fill=(65, 93, 165),
            outline=(25, 25, 25),
            width=max(2, round(1.4 * scale)),
        )
        dx, dy = label_offsets[generator]
        draw.text(
            (x + dx * canvas_width, y + dy * canvas_height),
            generator,
            fill=(35, 35, 35),
            font=font,
            anchor="mm",
        )


def render_paper_figure(
    system,
    low_words,
    output_path,
    width=1400,
    render_depth=12,
    rotation=0.0,
    reflect=False,
    include_diagram=True,
    style=RenderStyle(),
    emphasized_roots=(),
    draw_small_hyperplanes=True,
    secondary_words=(),
):
    started_at = time.perf_counter()
    if not isinstance(width, int) or width < 200:
        raise ValueError("The output width must be at least 200 pixels.")
    if not isinstance(render_depth, int) or render_depth < 1:
        raise ValueError("render_depth must be a positive integer.")

    scale = style.supersample
    canvas_width = width * scale
    canvas_height = round(width * 0.78) * scale
    disk_center = np.array((0.54 * canvas_width, 0.515 * canvas_height))
    disk_radius = min(0.368 * canvas_width, 0.465 * canvas_height)

    unique_low_words = {}
    unique_secondary_words = {}
    requested_chambers = []
    for word in low_words:
        chamber = system.canonicalize_word(word)
        requested_chambers.append(chamber)
        unique_low_words.setdefault(chamber.key, str(word))
    for word in secondary_words:
        chamber = system.canonicalize_word(word)
        requested_chambers.append(chamber)
        if chamber.key not in unique_low_words:
            unique_secondary_words.setdefault(chamber.key, str(word))
    display_words = dict(unique_low_words)
    display_words.update(unique_secondary_words)
    secondary_display_words = set(unique_secondary_words.values())
    maximum_requested_length = max(
        (chamber.length for chamber in requested_chambers),
        default=0,
    )
    effective_render_depth = max(render_depth, maximum_requested_length + 2)

    chambers = system.enumerate_chambers(effective_render_depth)
    roots = {}
    root_occurrences = {}
    wall_occurrences = []
    for chamber in chambers.values():
        matrix = chamber.matrix
        for index in range(3):
            root = system._positive_root(matrix @ system.simple_roots[index])
            key = system.root_key(root)
            roots[key] = root
            root_occurrences[key] = root_occurrences.get(key, 0) + 1
            generator = GENERATOR_ORDER[index]
            wall = chamber.walls[generator]
            wall_occurrences.append(
                RenderedWallOccurrence(
                    chamber_key=chamber.key,
                    chamber_word=chamber.word,
                    wall_type=generator,
                    root_key=wall.root_key,
                    reflection=wall.reflection.copy(),
                    neighbor_key=wall.neighbor_key,
                    curve_key=key,
                )
            )

    small_roots = system.enumerate_small_roots()
    for key, record in small_roots.items():
        roots[key] = record.coefficients
        root_occurrences.setdefault(key, 0)

    emphasized = {}
    for root in emphasized_roots:
        coefficients = (
            root.coefficients
            if isinstance(root, RootRecord)
            else np.asarray(root, dtype=np.float64).reshape(3)
        )
        key = system.root_key(coefficients)
        emphasized[key] = coefficients
        roots[key] = coefficients
        root_occurrences.setdefault(key, 0)

    def to_pixels(points):
        oriented = _orient(points, rotation, reflect)
        result = np.empty_like(oriented)
        result[..., 0] = disk_center[0] + disk_radius * oriented[..., 0]
        result[..., 1] = disk_center[1] - disk_radius * oriented[..., 1]
        return np.rint(result).astype(np.int32)

    sampled_roots = {
        key: to_pixels(poincare_geodesic(root, system).sample())
        for key, root in roots.items()
    }

    image = Image.new("RGB", (canvas_width, canvas_height), (255, 255, 255))
    draw = ImageDraw.Draw(image)
    grid_width = max(2, round(style.grid_width * scale))
    small_width = max(grid_width + 2, round(style.small_wall_width * scale))
    emphasized_width = max(
        grid_width + 2,
        round(style.emphasized_wall_width * scale),
    )
    curve_draw_counts = {key: 0 for key in sampled_roots}
    small_curve_draw_counts = (
        {key: 0 for key in small_roots}
        if draw_small_hyperplanes
        else {}
    )
    emphasized_curve_draw_counts = {key: 0 for key in emphasized}

    for key, points in sampled_roots.items():
        draw.line([tuple(point) for point in points], fill=style.grid_color, width=grid_width)
        curve_draw_counts[key] += 1
    disk_box = (
        disk_center[0] - disk_radius,
        disk_center[1] - disk_radius,
        disk_center[0] + disk_radius,
        disk_center[1] + disk_radius,
    )
    draw.ellipse(disk_box, outline=style.grid_color, width=grid_width)

    normals = _lorentz_normals(system)
    fundamental_center = _fundamental_center(system, normals)
    label_positions = {}
    seed_positions = {}
    for word in display_words.values():
        center = _word_center(system, word, normals, fundamental_center)
        seed = tuple(to_pixels(center).tolist())
        safe_seed = _nearest_white_pixel(image, seed)
        seed_positions[word] = safe_seed
        label_positions[word] = (
            round(safe_seed[0] / scale),
            round(safe_seed[1] / scale),
        )

    component_image = image.copy()
    component_markers = {}
    filled_components = 0
    primary_filled_components = 0
    secondary_filled_components = 0
    for index, (word, safe_seed) in enumerate(seed_positions.items(), start=1):
        marker = (index, 1, 254 - index)
        component_markers[word] = marker
        if component_image.getpixel(safe_seed) == (255, 255, 255):
            ImageDraw.floodfill(component_image, safe_seed, marker, thresh=0)
            filled_components += 1
            if word in secondary_display_words:
                secondary_filled_components += 1
            else:
                primary_filled_components += 1

    component_pixels = np.asarray(component_image)
    filled_pixels = np.asarray(image).copy()
    filled_pixels[component_pixels[:, :, 1] == 1] = style.fill_color
    for word in unique_secondary_words.values():
        marker = component_markers[word]
        secondary_component = np.all(component_pixels == marker, axis=2)
        filled_pixels[secondary_component] = style.secondary_fill_color
    identity_key = system.canonicalize_word("e").key
    identity_word = unique_low_words.get(identity_key)
    if identity_word is not None:
        identity_marker = component_markers[identity_word]
        identity_component = np.all(component_pixels == identity_marker, axis=2)
        filled_pixels[identity_component] = style.identity_fill_color
    image = Image.fromarray(filled_pixels, mode="RGB")

    draw = ImageDraw.Draw(image)
    for key, points in sampled_roots.items():
        draw.line([tuple(point) for point in points], fill=style.grid_color, width=grid_width)
        curve_draw_counts[key] += 1
    draw.ellipse(disk_box, outline=style.grid_color, width=grid_width)

    if draw_small_hyperplanes:
        for key in small_roots:
            points = sampled_roots[key]
            draw.line(
                [tuple(point) for point in points],
                fill=style.small_wall_color,
                width=small_width,
            )
            curve_draw_counts[key] += 1
            small_curve_draw_counts[key] += 1

    for key in emphasized:
        points = sampled_roots[key]
        draw.line(
            [tuple(point) for point in points],
            fill=style.emphasized_wall_color,
            width=emphasized_width,
        )
        curve_draw_counts[key] += 1
        emphasized_curve_draw_counts[key] += 1

    base_low_font_size = max(10, round(width * 11 / 650)) * scale
    minimum_low_font_size = max(scale, round(base_low_font_size * 0.12))
    context_font = _load_font(max(8, round(width * 8 / 650) * scale))
    label_background = np.asarray(image)
    low_fill_pixels = (
        np.all(label_background == style.fill_color, axis=2)
        | np.all(label_background == style.secondary_fill_color, axis=2)
        | np.all(label_background == style.identity_fill_color, axis=2)
    )
    blocked_pixels = ~low_fill_pixels
    blocked_integral = np.pad(
        blocked_pixels.astype(np.int64),
        ((1, 0), (1, 0)),
    ).cumsum(axis=0).cumsum(axis=1)
    low_labels_contained = True
    uncontained_labels = []
    label_font_sizes = {}
    label_rotations = {}
    for word, preferred_position in seed_positions.items():
        marker = component_markers[word]
        component_coordinates = np.argwhere(
            np.all(component_pixels == marker, axis=2)
        )
        stride = max(1, len(component_coordinates) // 800)
        component_coordinates = component_coordinates[::stride]
        candidate_positions = [preferred_position]
        candidate_positions.extend(
            (int(x), int(y))
            for y, x in component_coordinates
            if (int(x), int(y)) != preferred_position
        )
        candidate_positions.sort(
            key=lambda point: (
                (point[0] - preferred_position[0]) ** 2
                + (point[1] - preferred_position[1]) ** 2
            )
        )
        candidate_positions = np.asarray(candidate_positions, dtype=np.int32)

        chosen_font = None
        chosen_position = preferred_position
        chosen_rotation = 0
        font_sizes = list(range(
            base_low_font_size,
            minimum_low_font_size - 1,
            -max(1, scale),
        ))
        if not font_sizes or font_sizes[-1] != minimum_low_font_size:
            font_sizes.append(minimum_low_font_size)
        for font_size in font_sizes:
            candidate_font = _load_font(font_size)
            margin = max(1, min(scale, font_size // 4))
            text_box = draw.textbbox(
                (0, 0),
                word,
                font=candidate_font,
                anchor="mm",
            )
            for rotation in (0, 90):
                if rotation == 0:
                    left_offset, top_offset, right_offset, bottom_offset = text_box
                else:
                    text_width = text_box[2] - text_box[0]
                    text_height = text_box[3] - text_box[1]
                    left_offset = -(text_height + 1) // 2
                    right_offset = text_height // 2
                    top_offset = -(text_width + 1) // 2
                    bottom_offset = text_width // 2
                lefts = candidate_positions[:, 0] + left_offset - margin
                tops = candidate_positions[:, 1] + top_offset - margin
                rights = candidate_positions[:, 0] + right_offset + margin
                bottoms = candidate_positions[:, 1] + bottom_offset + margin
                in_bounds = (
                    (lefts >= 0)
                    & (tops >= 0)
                    & (rights <= canvas_width)
                    & (bottoms <= canvas_height)
                    & (rights > lefts)
                    & (bottoms > tops)
                )
                valid_indexes = np.flatnonzero(in_bounds)
                if valid_indexes.size:
                    left = lefts[valid_indexes]
                    top = tops[valid_indexes]
                    right = rights[valid_indexes]
                    bottom = bottoms[valid_indexes]
                    blocked_counts = (
                        blocked_integral[bottom, right]
                        - blocked_integral[top, right]
                        - blocked_integral[bottom, left]
                        + blocked_integral[top, left]
                    )
                    clear_indexes = valid_indexes[blocked_counts == 0]
                else:
                    clear_indexes = np.empty(0, dtype=np.int64)
                if clear_indexes.size:
                    chosen_font = candidate_font
                    chosen_position = tuple(
                        int(value)
                        for value in candidate_positions[clear_indexes[0]]
                    )
                    chosen_rotation = rotation
                    break
            if chosen_font is not None:
                break
        if chosen_font is None:
            chosen_font = _load_font(minimum_low_font_size)
            low_labels_contained = False
            uncontained_labels.append(word)
        label_positions[word] = (
            round(chosen_position[0] / scale),
            round(chosen_position[1] / scale),
        )
        label_font_sizes[word] = chosen_font.size / scale
        label_rotations[word] = chosen_rotation
        _draw_rotated_text(
            image,
            chosen_position,
            word,
            chosen_font,
            (35, 35, 35),
            chosen_rotation,
        )

    if include_diagram:
        _draw_coxeter_diagram(
            draw,
            system,
            canvas_width,
            canvas_height,
            scale,
        )

    high_resolution_pixels = np.asarray(image)
    blue_pixels = int(
        np.all(high_resolution_pixels == style.small_wall_color, axis=2).sum()
    )
    black_pixels = int(
        np.all(high_resolution_pixels == style.grid_color, axis=2).sum()
    )
    emphasized_pixels = int(
        np.all(
            high_resolution_pixels == style.emphasized_wall_color,
            axis=2,
        ).sum()
    )
    fill_pixels = int(
        np.all(high_resolution_pixels == style.fill_color, axis=2).sum()
    )
    identity_fill_pixels = int(
        np.all(
            high_resolution_pixels == style.identity_fill_color,
            axis=2,
        ).sum()
    )
    secondary_fill_pixels = int(
        np.all(
            high_resolution_pixels == style.secondary_fill_color,
            axis=2,
        ).sum()
    )

    def curve_has_color(points, color):
        clipped_x = np.clip(points[:, 0], 0, canvas_width - 1)
        clipped_y = np.clip(points[:, 1], 0, canvas_height - 1)
        sampled_colors = high_resolution_pixels[clipped_y, clipped_x]
        return bool(np.any(np.all(sampled_colors == color, axis=1)))

    visible_curve_keys = tuple(
        sorted(
            key
            for key, points in sampled_roots.items()
            if curve_has_color(
                points,
                (
                    style.emphasized_wall_color
                    if key in emphasized
                    else style.small_wall_color
                    if draw_small_hyperplanes and key in small_roots
                    else style.grid_color
                ),
            )
        )
    )
    visible_small_curve_keys = tuple(
        sorted(
            key
            for key in small_roots
            if draw_small_hyperplanes
            and curve_has_color(sampled_roots[key], style.small_wall_color)
        )
    )
    visible_emphasized_curve_keys = tuple(
        sorted(
            key
            for key in emphasized
            if curve_has_color(
                sampled_roots[key],
                style.emphasized_wall_color,
            )
        )
    )
    boundary_hits = 0
    for angle in np.linspace(0.0, 2.0 * pi, 720, endpoint=False):
        x = int(round(disk_center[0] + disk_radius * cos(angle)))
        y = int(round(disk_center[1] + disk_radius * sin(angle)))
        left = max(0, x - grid_width)
        right = min(canvas_width, x + grid_width + 1)
        top = max(0, y - grid_width)
        bottom = min(canvas_height, y + grid_width + 1)
        patch = high_resolution_pixels[top:bottom, left:right]
        if np.any(np.all(patch == style.grid_color, axis=2)):
            boundary_hits += 1

    final_height = round(width * 0.78)
    final_image = image.resize(
        (width, final_height),
        Image.Resampling.LANCZOS,
    )
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    final_image.save(output)
    elapsed = time.perf_counter() - started_at

    root_records = tuple(
        RenderedRootRecord(
            key=key,
            coefficients=roots[key].copy(),
            reflection=system.reflection_for_root(roots[key]),
            occurrences=root_occurrences[key],
            is_small=key in small_roots,
        )
        for key in sorted(roots)
    )
    diagram_outside_disk = bool(include_diagram) and _diagram_is_outside_disk(
        canvas_width,
        canvas_height,
        disk_center,
        disk_radius,
        scale,
    )

    return RenderResult(
        image_path=output,
        label_positions=label_positions,
        label_font_sizes=label_font_sizes,
        label_rotations=label_rotations,
        root_keys=tuple(sorted(roots)),
        small_root_keys=tuple(sorted(small_roots)),
        emphasized_root_keys=tuple(sorted(emphasized)),
        root_records=root_records,
        wall_occurrences=tuple(wall_occurrences),
        rendered_chamber_keys=tuple(sorted(chambers)),
        curve_draw_counts=curve_draw_counts,
        small_curve_draw_counts=small_curve_draw_counts,
        emphasized_curve_draw_counts=emphasized_curve_draw_counts,
        visible_curve_keys=visible_curve_keys,
        visible_small_curve_keys=visible_small_curve_keys,
        visible_emphasized_curve_keys=visible_emphasized_curve_keys,
        low_chamber_keys=tuple(sorted(unique_low_words)),
        secondary_chamber_keys=tuple(sorted(unique_secondary_words)),
        effective_render_depth=effective_render_depth,
        canvas_metrics={
            "blue_pixels": blue_pixels,
            "black_pixels": black_pixels,
            "emphasized_pixels": emphasized_pixels,
            "fill_pixels": fill_pixels,
            "secondary_fill_pixels": secondary_fill_pixels,
            "identity_fill_pixels": identity_fill_pixels,
            "boundary_sample_hits": boundary_hits,
            "diagram_requested": bool(include_diagram),
            "diagram_outside_disk": diagram_outside_disk,
            "low_labels_contained": low_labels_contained,
            "uncontained_labels": tuple(uncontained_labels),
            "filled_components": filled_components,
            "primary_filled_components": primary_filled_components,
            "secondary_filled_components": secondary_filled_components,
            "canvas_width": width,
            "canvas_height": final_height,
            "disk_radius": disk_radius / scale,
        },
        elapsed_seconds=elapsed,
    )


from dataclasses import dataclass


@dataclass(frozen=True)
class LayeredLowData:
    low_0_words: tuple[str, ...]
    low_1_words: tuple[str, ...]
    low_1_only_words: tuple[str, ...]


def _user_word_map(system, words, set_name):
    if not words:
        raise ValueError(f"{set_name} must not be empty.")
    result = {}
    for word in words:
        label = str(word).strip().lower()
        chamber = system.canonicalize_word(label)
        result.setdefault(chamber.key, label)
    return result


def prepare_zero_one_low_layers(system, low_0_words, low_1_words):
    """Validate the supplied complete L0 and L1, then form L1 minus L0."""
    low_0 = _user_word_map(system, low_0_words, "LOW_0_WORDS")
    low_1 = _user_word_map(system, low_1_words, "LOW_1_WORDS")
    missing_keys = set(low_0) - set(low_1)
    if missing_keys:
        missing_words = [low_0[key] for key in missing_keys]
        raise ValueError(
            "LOW_1_WORDS must be the complete L1 and include all of L0; missing: "
            + ", ".join(missing_words)
        )
    low_1_only = [word for key, word in low_1.items() if key not in low_0]
    return LayeredLowData(
        low_0_words=tuple(low_0.values()),
        low_1_words=tuple(low_1.values()),
        low_1_only_words=tuple(low_1_only),
    )


def parameter_slug(parameters):
    return "-".join(
        coxeter_order_label(normalize_coxeter_order(value)).replace("∞", "inf")
        for value in parameters
    )


def crop_white_border(image_path, margin=6):
    """Remove unused white canvas while retaining a small safe margin."""
    image = Image.open(image_path).convert("RGB")
    pixels = np.asarray(image)
    nonwhite = np.any(pixels < 250, axis=2)
    rows, columns = np.where(nonwhite)
    if not len(rows):
        return Path(image_path)
    left = max(int(columns.min()) - margin, 0)
    top = max(int(rows.min()) - margin, 0)
    right = min(int(columns.max()) + margin + 1, image.width)
    bottom = min(int(rows.max()) + margin + 1, image.height)
    image.crop((left, top, right, bottom)).save(image_path)
    return Path(image_path)


system = CoxeterSystem3(*COXETER_PARAMETERS)
layers = prepare_zero_one_low_layers(system, LOW_0_WORDS, LOW_1_WORDS)

output_file = OUTPUT_FILE or (
    f"shi-arrangement-{parameter_slug(COXETER_PARAMETERS)}.png"
)
style = RenderStyle(
    fill_color=DARK_PURPLE,
    secondary_fill_color=LIGHT_PURPLE,
)
result = render_paper_figure(
    system,
    layers.low_0_words,
    output_file,
    width=IMAGE_SIZE,
    render_depth=RENDER_DEPTH,
    include_diagram=INCLUDE_COXETER_DIAGRAM,
    style=style,
    secondary_words=layers.low_1_only_words,
)
crop_white_border(result.image_path)

print(f"L0: {len(layers.low_0_words)} elements")
print(f"L1 minus L0: {len(layers.low_1_only_words)} elements")
print(f"Image saved to: {Path(result.image_path).resolve()}")

try:
    from IPython.display import display
    display(Image.open(result.image_path))
except ImportError:
    pass
